# Day 076 — Exercise 2: analyze_screenshot, describe_screen, read_screen_text

**What you'll build:** The core vision function and two purpose-built wrappers.

**Why it matters:** `analyze_screenshot` is the bridge between a PIL Image and a vision LLM answer. All other vision tools delegate to it with a fixed prompt.

In [ ]:
from PIL import Image as _PILImage

def _make_mock_image(width=100, height=100, color=(100, 100, 100)):
    return _PILImage.new('RGB', (width, height), color=color)
_mock_analyze_fn    = lambda img, q: 'MOCK:' + q[:16]


## Task

1. `analyze_screenshot(image, question, analyze_fn=None) -> str`
   - Mock: `return analyze_fn(image, question)`
   - Real: convert PIL Image to base64 (BytesIO → PNG → b64encode), call `ollama.chat(model='llava', messages=[{role/content/images}])`, return `resp['message']['content']`

2. `describe_screen(image, analyze_fn=None) -> str`
   - Delegate to `analyze_screenshot` with prompt: `'Describe what you see on this screen in detail.'`

3. `read_screen_text(image, analyze_fn=None) -> str`
   - Delegate with: `'Extract all visible text from this image exactly as it appears.'`

## Your Implementation

In [ ]:
import io, base64

def analyze_screenshot(image, question, analyze_fn=None):
    """Ask a vision LLM a question about an image."""
    raise NotImplementedError

def describe_screen(image, analyze_fn=None):
    """Describe what is visible on screen."""
    raise NotImplementedError

def read_screen_text(image, analyze_fn=None):
    """Extract all visible text from the screen verbatim."""
    raise NotImplementedError


In [ ]:
import io, base64

def analyze_screenshot(image, question, analyze_fn=None):
    if analyze_fn is not None:
        return analyze_fn(image, question)
    import ollama
    buf = io.BytesIO()
    image.save(buf, format='PNG')
    img_b64 = base64.b64encode(buf.getvalue()).decode()
    resp = ollama.chat(
        model='llava',
        messages=[{'role': 'user', 'content': question, 'images': [img_b64]}],
    )
    return resp['message']['content']

def describe_screen(image, analyze_fn=None):
    return analyze_screenshot(
        image, 'Describe what you see on this screen in detail.',
        analyze_fn=analyze_fn)

def read_screen_text(image, analyze_fn=None):
    return analyze_screenshot(
        image, 'Extract all visible text from this image exactly as it appears.',
        analyze_fn=analyze_fn)


## Automated checks

In [ ]:

score, total = 0, 5
try:
    from PIL import Image as PILImage
    img = PILImage.new('RGB', (100, 100))

    result = analyze_screenshot(img, 'Test?', analyze_fn=_mock_analyze_fn)
    assert isinstance(result, str), f"expected str, got {type(result)}"
    score += 1; print("✅ analyze_screenshot returns str")

    captured = {}
    def _cap(i, q): captured.update(img=i, q=q); return 'CAPTURED'
    analyze_screenshot(img, 'My question', analyze_fn=_cap)
    assert captured.get('q') == 'My question' and captured.get('img') is img
    score += 1; print("✅ analyze_fn receives (image, question)")

    prompts = []
    describe_screen(img, analyze_fn=lambda i, q: (prompts.append(q), 'D')[1])
    assert prompts and ('Describe' in prompts[0] or 'describe' in prompts[0])
    score += 1; print("✅ describe_screen passes a describe prompt")

    d = describe_screen(img, analyze_fn=_mock_analyze_fn)
    assert isinstance(d, str)
    score += 1; print("✅ describe_screen returns str")

    tprompts = []
    read_screen_text(img, analyze_fn=lambda i, q: (tprompts.append(q), 'T')[1])
    assert tprompts and any(w in tprompts[0].lower() for w in ('text', 'extract'))
    score += 1; print("✅ read_screen_text passes a text-extraction prompt")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
import io, base64

def analyze_screenshot(image, question, analyze_fn=None):
    if analyze_fn is not None:
        return analyze_fn(image, question)
    import ollama
    buf = io.BytesIO()
    image.save(buf, format='PNG')
    img_b64 = base64.b64encode(buf.getvalue()).decode()
    resp = ollama.chat(
        model='llava',
        messages=[{'role': 'user', 'content': question, 'images': [img_b64]}],
    )
    return resp['message']['content']

def describe_screen(image, analyze_fn=None):
    return analyze_screenshot(
        image, 'Describe what you see on this screen in detail.',
        analyze_fn=analyze_fn)

def read_screen_text(image, analyze_fn=None):
    return analyze_screenshot(
        image, 'Extract all visible text from this image exactly as it appears.',
        analyze_fn=analyze_fn)
```

**Why inline the base64 conversion?** `screen_agent.py` is a standalone module. Inlining avoids importing from Day 67's path and keeps the module self-contained. The logic is identical to Day 67's `image_to_base64`.

</details>